# Lab 9: Data Cleaning - Real Estate Market Monitor

This notebook demonstrates the full data cleaning process implemented in Lab 9. Data cleaning is the process of correcting or removing wrong, incomplete, or inconsistent values so that the data is trustworthy before it is used for analysis or modelling.

**Objectives:**
- Analyse missing value patterns
- Handle missing values correctly using dropna, fillna, and medians
- Clean string columns by removing whitespace, normalising case, and fixing format
- Use regular expressions to clean complex text patterns
- Remove duplicate rows using exact and subset-based matching
- Convert columns to the correct data type
- Validate cleaned data with assert statements

In [ ]:
import sys
import os
import pandas as pd
import numpy as np

# Add src to path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../src')))

from cleaning.missing_handler import identify_missing_values, handle_missing_values, generate_missing_value_report
from cleaning.string_cleaner import clean_string_columns, normalize_property_types
from cleaning.deduplicator import remove_duplicates, count_duplicates_helper
from cleaning.type_converter import convert_types, get_memory_usage_report
from cleaning.validator import validate_data
from analytics.regex_ops import extended_regex_cleaning
from utils.logger import logging

import warnings
warnings.filterwarnings('ignore')

## 1. Load Data
As per the requirements, we load the raw CSV file produced in Lab 8.

In [ ]:
raw_csv_path = "../data/processed/analytics/raw_integrated_data.csv"
if os.path.exists(raw_csv_path):
    df = pd.read_csv(raw_csv_path)
    print(f"Loaded {len(df)} records from {raw_csv_path}")
else:
    print("Raw CSV not found. Please run the pipeline first. Generating synthetic data for demonstration...")
    from analytics.data_loader import get_integrated_data
    df = get_integrated_data()

df.head()

## 2. Identify Missing Values
We identify missing value patterns and save a report for later use in the cleaning pipeline.

In [ ]:
missing_report = identify_missing_values(df)
print("Missing Value Summary:")
print(missing_report)
generate_missing_value_report(df, output_path="../data/processed/cleaned/missing_report.csv")

## 3. String Cleaning and Regex Operations
Standardizing text columns and using regex for complex patterns. We work on a copy to keep original data unchanged.

In [ ]:
clean_df = df.copy()
clean_df = clean_string_columns(clean_df)
clean_df = normalize_property_types(clean_df)
# Regex operations: Extracting years, validating dates/languages, flagging short descriptions
clean_df = extended_regex_cleaning(clean_df)

print("Preview of cleaned text columns and regex results:")
cols_to_show = ['listing_id', 'description', 'type', 'short_description_flag']
print(clean_df[[c for c in cols_to_show if c in clean_df.columns]].head())

## 4. Deduplication
Removing exact and key duplicates (e.g., listing_id, or same title + date).

In [ ]:
print(f"Rows before deduplication: {len(clean_df)}")
count_duplicates_helper(clean_df)
clean_df = remove_duplicates(clean_df)
print(f"Rows after deduplication: {len(clean_df)}")

## 5. Handling Remaining Missing Values
Filling medians and placeholders.

In [ ]:
clean_df = handle_missing_values(clean_df)
print(f"Missing values after handling: {clean_df.isnull().sum().sum()}")

## 6. Type Conversion and Memory Optimization
Converting to appropriate data types and reporting memory usage.

In [ ]:
print("Memory usage before conversion:")
get_memory_usage_report(clean_df)
clean_df = convert_types(clean_df)
print("\nMemory usage after conversion:")
get_memory_usage_report(clean_df)

## 7. Data Validation
Final check using assertions to confirm the data meets all requirements.

In [ ]:
is_valid = validate_data(clean_df)
print(f"Validation Result: {'SUCCESS' if is_valid else 'FAILED'}")

## 8. Export Final Cleaned Dataset

In [ ]:
output_path = "../data/processed/cleaned/cleaned_data.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
clean_df.to_csv(output_path, index=False)
print(f"Cleaned dataset saved as {output_path}")